# 02 — Data Preprocessing & Feature Engineering

**职责**: 数据清洗 → 缺失值处理 → 特征构建 → 特征选择 → 输出处理后的特征矩阵

**输入**: `data/raw/train_data.csv`, `data/raw/test_data.csv`  
**输出**: `data/processed/train_processed.parquet`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np

from config import DATA_RAW, DATA_TEST, DATA_PROC, TARGET
from src.preprocessing import load_and_clean, fill_missing, build_feature_matrix
from src.feature_engineering import select_features, compute_ensemble_importance

## 1. 加载原始数据 & 基础清洗

In [ ]:
df = load_and_clean(DATA_RAW)
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

## 2. 缺失值处理

检查填充后的数据质量。如果需要更精细的策略，可以修改 `src/preprocessing.py` 中的 `fill_missing()`。

In [ ]:
missing_after = df.isnull().sum()
print(f"Remaining missing columns: {(missing_after > 0).sum()}")
if (missing_after > 0).any():
    print(missing_after[missing_after > 0].sort_values(ascending=False).head(10))

## 3. 特征选择

使用 Lasso + ElasticNet + RandomForest 集成方法筛选最优特征子集。

In [ ]:
# TODO: 根据 EDA 结论调整 top_n（EDA 学习曲线的拐点）
X, y, selected_features = select_features(df, TARGET, top_n=30)
print(f"Selected {len(selected_features)} features")
print(f"X shape: {X.shape}, y shape: {y.shape}")

## 4. 保存处理后的数据

In [ ]:
DATA_PROC.parent.mkdir(parents=True, exist_ok=True)

processed = X.copy()
processed[TARGET] = y.values
processed.to_parquet(DATA_PROC, index=False)
print(f"Saved processed data to {DATA_PROC}")
print(f"Final shape: {processed.shape}")